In [25]:
import pandas as pd
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text

df = pd.read_csv("IMDB Dataset.csv", engine="python", on_bad_lines="skip")

df = df.sample(20000, random_state=42)

df["review"] = df["review"].apply(clean_text)
df["sentiment"] = df["sentiment"].map({"positive":1, "negative":0})

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df["review"])
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = MLPClassifier(
    hidden_layer_sizes=(64,32),
    max_iter=100,
    random_state=42,
    early_stopping=True
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# test sample
sample = ["I love this movie. It is amazing!"]
sample_clean = [clean_text(s) for s in sample]
sample_vec = vectorizer.transform(sample_clean)

print("Sample prediction:", model.predict(sample_vec))

# save
joblib.dump(model, "nn_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

Accuracy: 0.87775
              precision    recall  f1-score   support

           0       0.87      0.88      0.88      1998
           1       0.88      0.87      0.88      2002

    accuracy                           0.88      4000
   macro avg       0.88      0.88      0.88      4000
weighted avg       0.88      0.88      0.88      4000

Sample prediction: [1]


['vectorizer.pkl']

In [26]:
df_sample = df.sample(10000, random_state=42)
df_sample.to_csv("imdb.csv", index=False)

In [27]:
from google.colab import files
files.download("imdb.csv")
#files.download("nn_model.pkl")
#files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>